# Follow-up Experiments (post E0-E8) - paper-hardening pack

Three cheap experiments that close the remaining gaps found in the manuscript
review. **All run on a CPU (High-RAM) runtime** - they reuse Drive caches from
E0-E8 and spend no GPU units. Run SETUP first, then the three cells in any
order (E8c stages the val images, ~10 min the first time).

| Cell | What it adds to the paper | Wall-clock |
|------|---------------------------|------------|
| E7c | Tests the paper's headline claim against the missing baseline: is the E3 answerability-triage score (alone / combined with confidence / full stack) a better risk ranker than VQA confidence? Also applies Benjamini-Hochberg FDR across ALL E7/E7b/E7c significance tests. | ~10-20 min (bootstrap) |
| E5b | Refusal-gated ARR/FRR at 90/80/70% coverage (thresholds from cal only). Replaces the misleading ungated FRR~0.78 with deployment-aligned numbers for Sec. VI-D. | ~2 min |
| E8c | F9 qualitative grid (QUAL_SEED=7, rule-sampled): answered / high-confidence-wrong danger / good refusal with retake action / wrong-reason refusal. Plus a JSON manifest so captions can quote examples. | ~15 min (stages val images) |

After these finish, download `results/E7c_risk_signals/`, `results/E5b_gated_recovery/`,
`results/E8c_qualitative/`, and `results/figures/F9_*` from Drive and paste the
E7c summary here - the manuscript numbers get updated based on what E7c finds.


## SETUP - clone repo + minimal deps *(same cell as the master notebook)*

In [ ]:
# ====================================================================
#  SETUP - run FIRST on every fresh runtime (any runtime type).
#  Clones/updates the repo and installs ONLY missing packages.
#  Never reinstalls or downgrades anything Colab already ships
#  (that is what used to break the NumPy/pandas binary stack).
#  Re-running on a warm runtime finishes in seconds.
# ====================================================================
import importlib.util, os, subprocess, sys

REPO_URL  = 'https://github.com/meteorboyF/VQA-paper.git'
REPO_ROOT = '/content/VQA-paper'

# ── Optional switches (set BEFORE anything imports src) ──────────────
# os.environ['VQA_FORCE_RERUN'] = '1'                      # ignore all caches/DONE markers
# os.environ['VQA_BACKBONES']   = 'clip,mobilenet,dinov2'  # full 3-backbone table (A100 recommended)
# os.environ['VQA_DRIVE_BASE']  = '/content/drive/MyDrive/VQA_ML/AVA_VizWiz'  # if your Drive layout differs

def sh(args, check=False):
    print('$', ' '.join(args))
    proc = subprocess.run(args, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {args}')
    return proc

# 1) Clone or update to the latest push on main
if not os.path.exists(REPO_ROOT):
    sh(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)
else:
    if sh(['git', '-C', REPO_ROOT, 'pull', '--ff-only']).returncode != 0:
        print('[setup] pull failed; hard-resetting the (disposable) clone to origin/main')
        sh(['git', '-C', REPO_ROOT, 'fetch', 'origin', 'main'], check=True)
        sh(['git', '-C', REPO_ROOT, 'reset', '--hard', 'origin/main'], check=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
head = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'[setup] repo at {REPO_ROOT}, HEAD={head}')

# 2) Install ONLY what is missing (import-probe first, zero pip on warm runtimes)
NEEDED = {  # import name -> pip spec
    'torch': 'torch', 'torchvision': 'torchvision',
    'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow',
    'sklearn': 'scikit-learn', 'scipy': 'scipy',
    'matplotlib': 'matplotlib', 'PIL': 'Pillow', 'tqdm': 'tqdm',
    'transformers': 'transformers>=4.44',
    'open_clip': 'open-clip-torch>=2.26',
    'timm': 'timm>=1.0',
    'einops': 'einops', 'ftfy': 'ftfy', 'regex': 'regex',
}
missing = [spec for mod, spec in NEEDED.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print('[setup] installing missing packages:', missing)
    sh([sys.executable, '-m', 'pip', 'install', '-q', '--progress-bar', 'off',
        *missing], check=True)
else:
    print('[setup] all packages already present - no pip work needed.')

# 3) Safety net: verify the compiled numeric stack imports; repair only if broken
r = sh([sys.executable, 'scripts/colab_preflight.py'])
if r.returncode == 10:
    raise SystemExit('Numeric stack was repaired. Runtime -> Restart runtime, '
                     'then rerun this SETUP cell before continuing.')
if r.returncode != 0:
    raise RuntimeError('Numeric stack check failed - see output above.')

print('[setup] READY. Run the next cells - finished experiments skip themselves.')


## E7c - Risk-signal baselines + BH-FDR  *(CPU)*
If nothing beats confidence, the claim upgrades to: strongest among all signals evaluated, **including our own triage head**. If the combined model wins, the paper gets a stronger (positive) result instead.

In [ ]:
# ====================================================================
#  E7c - Risk-signal baselines for selective prediction
#  RUNTIME: CPU High-RAM. Cached data only, no GPU.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e7c_risk_signals
e7c_risk_signals.main()


## E5b - Refusal-gated ARR/FRR  *(CPU)*
Deployment-aligned recovery metrics: retake advice only counts after the confidence gate refuses. Thresholds selected on cal at 90/80/70% coverage.

In [ ]:
# ====================================================================
#  E5b - Refusal-gated actionable recovery
#  RUNTIME: CPU High-RAM. Cached data only, no GPU.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e5b_gated_recovery
e5b_gated_recovery.main()


## E8c - Qualitative figure F9  *(CPU; stages val images ~3.3 GB)*
Rule-sampled example grid incl. the high-confidence-WRONG danger panel. Output: `results/figures/F9_qualitative_grid.{pdf,png}` + manifest JSON.

In [ ]:
# ====================================================================
#  E8c - Qualitative F9 grid + manifest
#  RUNTIME: CPU High-RAM. Stages val images only; no GPU.
#  Skips itself instantly if already completed (DONE marker on Drive).
# ====================================================================
import os, sys
REPO_ROOT = '/content/VQA-paper'
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from src.experiments import e8c_qualitative
e8c_qualitative.main()
